# Fourier Series Visualizer v0.2 만들기

이 노트북은 **전체 앱 소스**, 수학 설명, 프로젝트 파일 생성, 검증 예제를 담습니다.
위에서부터 실행하면 노트북 옆 `fourier_visualizer` 폴더에 세 파일을 생성합니다.
이미 생성된 프로젝트는 노트북 실행 없이도 바로 사용할 수 있습니다.

## 1. Fourier Series의 표기

$$f(x)\sim a_0+\sum_{n=1}^{\infty}[a_n\cos(n\pi x/L)+b_n\sin(n\pi x/L)]$$
$$a_0=\frac{1}{2L}\int_{-L}^{L}f(x)dx$$
$$a_n=\frac{1}{L}\int_{-L}^{L}f(x)\cos(n\pi x/L)dx,\quad b_n=\frac{1}{L}\int_{-L}^{L}f(x)\sin(n\pi x/L)dx$$

여기서는 상수항이 `a0`입니다. `a0/2` 표기와 혼동하지 않습니다.
구간의 양 끝점을 포함해 표본을 만들고 사다리꼴 적분으로 계수를 근사합니다.
주기적 확장의 불연속점에서는 급수가 좌우 극한의 평균으로 수렴할 수 있습니다.

## 2. 생성 경로

기본 Jupyter 작업 디렉터리 또는 이 과목의 상위 작업 디렉터리를 지원합니다.
다른 위치에서 실행한다면 `notebook_directory`만 노트북 폴더로 지정하세요.
파일 생성 셀은 동일 이름의 프로젝트 파일을 현재 노트북 소스로 갱신합니다.


In [ ]:
from pathlib import Path

notebook_directory = Path.cwd()
if (notebook_directory / "공업수학2" / "만들기.ipynb").exists():
    notebook_directory = notebook_directory / "공업수학2"
project_directory = notebook_directory / "fourier_visualizer"
project_directory.mkdir(parents=True, exist_ok=True)
print("프로젝트 경로:", project_directory)


## 3. 전체 app.py 코드

다음 셀은 파일로 저장할 전체 소스를 문자열로 정의합니다. 실행 시 UI를 바로 띄우지 않습니다.

- **입력/샘플링:** 문법 허용 목록 → `sympify` → `lambdify` → 실수 배열 검사
- **수학 함수:** 계수와 부분합은 별도 함수이며, UI에 의존하지 않습니다.
- **그래프 함수:** Figure를 반환하므로 화면 배치나 추후 애니메이션을 바꾸기 쉽습니다.
- **UI:** 입력을 한 번 읽어 필요한 최대 차수까지 계산하고 결과를 재사용합니다.

`max([current_N, *comparison_N])`는 빈 비교 목록과 현재 차수가 더 큰 경우를 모두 처리합니다.
상수 함수는 `broadcast_to`로 표본 배열 크기에 맞추고, NaN/inf는 오류 메시지로 처리합니다.


In [ ]:
# 아래 문자열 전체가 실행 가능한 app.py 소스입니다.
app_source = r'''
"""Fourier Series Visualizer v0.2: numerical mathematics and Streamlit UI."""

import ast
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st
import sympy as sp


# ── 1. 안전한 수식 입력과 샘플링 ──────────────────────────────────────
# sympify는 임의의 Python 코드를 위한 샌드박스가 아니므로,
# 허용한 이름과 산술 문법만 통과시킨 후 호출한다.
X = sp.Symbol("x", real=True)
ALLOWED_NAMES = {
    "x": X, "pi": sp.pi, "E": sp.E,
    "sin": sp.sin, "cos": sp.cos, "tan": sp.tan,
    "exp": sp.exp, "log": sp.log, "sqrt": sp.sqrt,
    "abs": sp.Abs, "Abs": sp.Abs, "sign": sp.sign,
    "sinh": sp.sinh, "cosh": sp.cosh, "tanh": sp.tanh,
}
ALLOWED_NODES = (
    ast.Expression, ast.BinOp, ast.UnaryOp, ast.Call, ast.Name,
    ast.Load, ast.Constant, ast.Add, ast.Sub, ast.Mult, ast.Div,
    ast.Pow, ast.UAdd, ast.USub,
)


def parse_expression(expression_text):
    """허용 목록을 검사한 문자열을 SymPy 수식으로 변환한다."""
    if not expression_text.strip() or len(expression_text) > 300:
        raise ValueError("수식을 1~300자 이내로 입력하세요.")
    # 이름, 숫자, 괄호, 기본 연산자 외에는 허용하지 않는다.
    if not re.fullmatch(r"[A-Za-z0-9_+\-*/().,\s]+", expression_text):
        raise ValueError("지원하지 않는 문자가 있습니다. 거듭제곱은 **로 입력하세요.")
    tree = ast.parse(expression_text, mode="eval")
    for node in ast.walk(tree):
        if not isinstance(node, ALLOWED_NODES):
            raise ValueError("기본 산술 연산과 지원 함수만 사용할 수 있습니다.")
        if isinstance(node, ast.Name) and node.id not in ALLOWED_NAMES:
            raise ValueError(f"지원하지 않는 이름: {node.id}")
        if isinstance(node, ast.Constant):
            if type(node.value) not in (int, float):
                raise ValueError("숫자 상수만 사용할 수 있습니다.")
        if isinstance(node, ast.Call):
            if (not isinstance(node.func, ast.Name)
                    or node.func.id not in ALLOWED_NAMES
                    or not callable(ALLOWED_NAMES[node.func.id])
                    or len(node.args) != 1 or node.keywords):
                raise ValueError("지원 함수에는 인자 하나만 입력하세요.")
    expression = sp.sympify(expression_text, locals=ALLOWED_NAMES)
    if not isinstance(expression, sp.Expr) or expression.free_symbols - {X}:
        raise ValueError("x에 대한 실수 함수를 입력하세요.")
    return expression


def sample_function(expression, x):
    """상수 함수도 x와 같은 크기로 확장하고 비유한/복소 값을 거부한다."""
    numpy_function = sp.lambdify(X, expression, modules="numpy")
    with np.errstate(all="ignore"):
        values = np.asarray(numpy_function(x))
    if np.iscomplexobj(values):
        raise ValueError("복소수 값이 발생했습니다. 실수 함수를 입력하세요.")
    values = np.array(np.broadcast_to(values, x.shape), dtype=float, copy=True)
    if not np.all(np.isfinite(values)):
        raise ValueError("샘플에서 NaN 또는 inf가 발생했습니다. 함수와 구간을 확인하세요.")
    return values


# ── 2. UI와 독립적인 Fourier 계산 ─────────────────────────────────────
def calculate_fourier_coefficients(x, y, L, max_N):
    """[-L,L] 표본에 사다리꼴 적분을 적용한다. an[0]은 a_1이다."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if not np.isfinite(L) or L <= 0:
        raise ValueError("L은 유한한 양수여야 합니다.")
    if not isinstance(max_N, (int, np.integer)) or max_N < 1:
        raise ValueError("max_N은 양의 정수여야 합니다.")
    if x.ndim != 1 or x.size < 2 or x.shape != y.shape:
        raise ValueError("x와 y는 길이가 같은 1차원 표본이어야 합니다.")
    if not np.all(np.isfinite(x)) or not np.all(np.isfinite(y)):
        raise ValueError("표본은 모두 유한해야 합니다.")
    if not np.all(np.diff(x) > 0):
        raise ValueError("x는 오름차순이어야 합니다.")
    if not np.isclose(x[0], -L) or not np.isclose(x[-1], L):
        raise ValueError("표본 구간은 [-L, L]이어야 합니다.")

    # NumPy 2.x의 API를 우선 사용하고 이전 버전도 지원한다.
    integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    cosine_coefficients = np.zeros(max_N)
    sine_coefficients = np.zeros(max_N)
    with np.errstate(over="raise", invalid="raise", divide="raise"):
        a0 = float(integrate(y, x=x) / (2 * L))
        for n in range(1, max_N + 1):
            angle = n * np.pi * (x / L)
            cosine_coefficients[n - 1] = integrate(y * np.cos(angle), x=x) / L
            sine_coefficients[n - 1] = integrate(y * np.sin(angle), x=x) / L
    if not np.all(np.isfinite(np.r_[a0, cosine_coefficients, sine_coefficients])):
        raise ValueError("계수가 유한하지 않습니다. 함수의 크기를 줄여 주세요.")
    return a0, cosine_coefficients, sine_coefficients


def calculate_fourier_sum(x, L, a0, an, bn, N):
    """a0 + Σ(an cos(nπx/L) + bn sin(nπx/L))를 계산한다."""
    if not np.isfinite(L) or L <= 0:
        raise ValueError("L은 유한한 양수여야 합니다.")
    if not isinstance(N, (int, np.integer)) or not 0 <= N <= min(len(an), len(bn)):
        raise ValueError("N에 필요한 Fourier 계수가 부족합니다.")
    x = np.asarray(x, dtype=float)
    result = np.full_like(x, a0, dtype=float)
    with np.errstate(over="raise", invalid="raise"):
        for n in range(1, N + 1):
            angle = n * np.pi * (x / L)
            result += an[n - 1] * np.cos(angle) + bn[n - 1] * np.sin(angle)
    if not np.all(np.isfinite(result)):
        raise ValueError("부분합에서 비유한 값이 발생했습니다.")
    return result


def calculate_errors(y, approximation):
    """같은 표본에서의 오차: 연속 구간 전체의 최대 오차와는 구별된다."""
    with np.errstate(over="raise", invalid="raise"):
        error = y - approximation
        mse = float(np.mean(error ** 2))
        rmse = float(np.sqrt(mse))
        maximum_error = float(np.max(np.abs(error)))
    return error, {"MSE": mse, "RMSE": rmse, "Maximum Absolute Error": maximum_error}


# ── 3. 그래프 생성: Figure 반환으로 화면 로직과 분리 ───────────────────
def make_comparison_figure(x, y, approximations, title):
    figure, axes = plt.subplots(figsize=(7, 4))
    axes.plot(x, y, color="black", linewidth=2, label="f(x)")
    for order, approximation in approximations.items():
        axes.plot(x, approximation, linewidth=1.4, label=f"N = {order}")
    axes.set(title=title, xlabel="x", ylabel="Function value")
    axes.grid(True, alpha=0.3)
    axes.legend()
    figure.tight_layout()
    return figure


def make_error_figure(x, error):
    figure, axes = plt.subplots(figsize=(12, 3))
    axes.plot(x, error, label="f(x) - S_N(x)", color="tab:red")
    axes.axhline(0, color="gray", linewidth=0.8)
    axes.set(title="Approximation error", xlabel="x", ylabel="Error")
    axes.grid(True, alpha=0.3)
    axes.legend()
    figure.tight_layout()
    return figure


def make_spectrum_figure(an, bn):
    orders = np.arange(1, len(an) + 1)
    figure, axes = plt.subplots(figsize=(10, 3.5))
    axes.stem(orders, np.abs(an), linefmt="C0-", markerfmt="C0o",
              basefmt=" ", label="|a_n|")
    axes.stem(orders, np.abs(bn), linefmt="C1--", markerfmt="C1x",
              basefmt=" ", label="|b_n|")
    axes.set(title="Coefficient spectrum", xlabel="n", ylabel="Coefficient magnitude")
    axes.grid(True, alpha=0.3)
    axes.legend()
    figure.tight_layout()
    return figure


def show_figure(figure):
    """표시 후 Figure를 닫아 Streamlit 재실행 시 메모리 누적을 막는다."""
    try:
        st.pyplot(figure)
    finally:
        plt.close(figure)


# ── 4. Streamlit 화면 ────────────────────────────────────────────────
def main():
    st.set_page_config(page_title="Fourier Series Visualizer v0.2", layout="wide")
    st.title("Fourier Series Visualizer v0.2")
    st.write("[-L, L]에서 정의된 함수의 주기적 확장과 Fourier 부분합을 살펴봅니다.")
    st.latex(r"f(x) \sim a_0 + \sum_{n=1}^{\infty}\left[a_n\cos\frac{n\pi x}{L}+b_n\sin\frac{n\pi x}{L}\right]")

    with st.sidebar:
        st.header("함수 및 계산 설정")
        expression_text = st.text_input("f(x)", value="x")
        st.caption("예: x, x**2, sin(x), cos(x), exp(x), abs(x), sign(x), 3")
        length_text = st.text_input("L (양수, 기본값 pi)", value="pi")
        current_N = st.slider("현재 Fourier 차수 N", 1, 100, 10)
        num_points = st.slider("샘플링 점 개수", 500, 20000, 5000, step=100)
        comparison_N = st.multiselect("비교할 N", [1, 3, 5, 10, 20, 50], default=[1, 3, 5, 10])

    try:
        expression = parse_expression(expression_text)
        length_expression = parse_expression(length_text)
        if length_expression.free_symbols:
            raise ValueError("L에는 x가 없는 양수 상수를 입력하세요.")
        L = float(length_expression)
        if not np.isfinite(L) or L <= 0 or not np.isfinite(2 * L):
            raise ValueError("L 및 주기 2L은 유한한 양수여야 합니다.")
        x = np.linspace(-L, L, num_points)
        y = sample_function(expression, x)
        # 빈 비교 목록도 안전하고 현재 N이 더 커도 충분히 계산한다.
        max_N = max([current_N, *comparison_N])
        a0, an, bn = calculate_fourier_coefficients(x, y, L, max_N)
        orders = sorted(set([current_N, *comparison_N]))
        approximations = {
            order: calculate_fourier_sum(x, L, a0, an, bn, order)
            for order in orders
        }
        error, metrics = calculate_errors(y, approximations[current_N])
        convergence_rows = []
        for order in sorted(comparison_N):
            _, order_metrics = calculate_errors(y, approximations[order])
            convergence_rows.append({"N": order, "MSE": order_metrics["MSE"], "RMSE": order_metrics["RMSE"]})
    except Exception as exc:
        st.error(f"입력 또는 수치 계산을 확인하세요: {exc}")
        return

    left, right = st.columns(2)
    with left:
        st.subheader(f"현재 N = {current_N}의 Fourier approximation")
        show_figure(make_comparison_figure(x, y, {current_N: approximations[current_N]}, "Current approximation"))
    with right:
        st.subheader("여러 N의 convergence comparison")
        if not comparison_N:
            st.info("비교할 N을 선택하면 부분합이 추가됩니다.")
        show_figure(make_comparison_figure(x, y, {n: approximations[n] for n in sorted(comparison_N)}, "Convergence comparison"))

    for column, (name, value) in zip(st.columns(3), metrics.items()):
        column.metric(name, f"{value:.6g}")
    st.caption("오차는 양 끝점을 포함한 표본에서 계산합니다. 불연속점의 값과 주기 경계 때문에 최대 오차가 0으로 수렴하지 않을 수 있습니다.")
    st.subheader("Error graph")
    show_figure(make_error_figure(x, error))

    st.subheader("Fourier coefficient table")
    st.write(f"a0 = {a0:.12g}")
    st.caption(f"현재 N과 비교 N에 필요한 계수 n = 1, …, {max_N}을 표시합니다.")
    coefficient_table = pd.DataFrame({
        "n": np.arange(1, max_N + 1), "a_n": an, "b_n": bn,
        "|a_n|": np.abs(an), "|b_n|": np.abs(bn),
    })
    st.dataframe(coefficient_table, hide_index=True)
    st.subheader("Coefficient spectrum")
    show_figure(make_spectrum_figure(an, bn))

    st.subheader("N별 수렴 오차 비교")
    st.dataframe(pd.DataFrame(convergence_rows, columns=["N", "MSE", "RMSE"]), hide_index=True)
    st.caption("sign(x)를 입력하고 N을 증가시키면 x = 0 및 주기 경계 근처의 Gibbs 진동을 관찰할 수 있습니다. 진동 영역은 좁아지지만 overshoot는 남습니다.")

    st.divider()
    st.subheader("현재 함수 정보")
    st.latex("f(x) = " + sp.latex(expression))
    st.write(f"구간: [{-L:.8g}, {L:.8g}] · period = 2L = {2 * L:.8g} · current N = {current_N}")


if __name__ == "__main__":
    main()
'''

app_path = project_directory / 'app.py'
app_path.write_text(app_source.lstrip(), encoding='utf-8')
print('생성:', app_path)


## 4. 라이브러리 의존성

NumPy 2.x의 `trapezoid`를 사용합니다. Matplotlib은 그래프, pandas는 표,
SymPy는 수식 해석, Streamlit은 인터랙티브 화면을 담당합니다.


In [ ]:
requirements_source = '''streamlit>=1.32
numpy>=2.0
matplotlib>=3.9
pandas>=2.2.2
sympy>=1.13
'''
(project_directory / 'requirements.txt').write_text(requirements_source, encoding='utf-8')


## 5. README 생성

설치, 수학 정의, 기능, 테스트 예제, 향후 확장 계획을 프로젝트에 함께 보관합니다.

In [ ]:
readme_source = r'''# Fourier Series Visualizer v0.2

공업수학의 Fourier Series를 실험하는 Python/Streamlit 학습 앱입니다.
Python 3.10 이상을 권장합니다.

## 설치 및 실행

터미널에서 이 README가 있는 `fourier_visualizer` 폴더로 이동한 뒤 실행합니다.

```bash
pip install -r requirements.txt
streamlit run app.py
```

브라우저에 표시되는 로컬 주소에서 사용합니다. `python app.py`가 아니라
`streamlit run app.py`로 실행해야 Streamlit 화면이 동작합니다.
상위 폴더의 `만들기.ipynb`에는 설명, 파일 생성 코드, 수학 검증 코드가 있습니다.

## Fourier Series 정의

주기 2L인 함수에 대해 다음 표기를 사용합니다. 상수항은 **a0이며 a0/2가 아닙니다.**

\[
f(x) \sim a_0 + \sum_{n=1}^{\infty}\left[a_n\cos(n\pi x/L)+b_n\sin(n\pi x/L)\right]
\]
\[
a_0=\frac{1}{2L}\int_{-L}^{L}f(x)\,dx,\quad
a_n=\frac{1}{L}\int_{-L}^{L}f(x)\cos(n\pi x/L)\,dx,\quad
b_n=\frac{1}{L}\int_{-L}^{L}f(x)\sin(n\pi x/L)\,dx.
\]

수치 계수는 양 끝점을 포함하는 균일 표본과 `numpy.trapezoid()`로 계산합니다.
`an[n-1]`, `bn[n-1]`은 각각 n번째 계수입니다.
불연속점에서 급수는 통상 좌우 극한의 평균으로 수렴하므로 모든 점에서
원래 함수값과 같다는 의미는 아닙니다.

## 주요 기능

- 문자열 함수 입력, L 설정, N=1~100, 표본 수 500~20000
- 현재 부분합과 여러 N의 부분합을 두 열로 비교
- MSE, RMSE, Maximum Absolute Error 및 오차 그래프
- 계수 표, 별도의 a0 표시, |a_n| 및 |b_n| stem plot
- N별 MSE/RMSE 표와 현재 함수 정보
- 상수 함수의 배열 확장, 잘못된 수식 및 NaN/inf/복소 값 처리
- 비교 목록이 비어 있거나 현재 N이 비교 차수보다 큰 경우 지원

입력은 `sympy.sympify()`와 `sympy.lambdify()`로 변환합니다.
기본 산술 및 지원 함수만 허용하며 거듭제곱은 `**`로 입력합니다.
지원 예: sin, cos, tan, exp, log, sqrt, abs, sign, sinh, cosh, tanh.
이 앱은 로컬 학습용입니다. 수식 허용 목록은 계산 시간이나 메모리 사용량의
상한을 보장하지 않으므로 불특정 다수에게 배포하려면 별도 자원 제한이 필요합니다.

## 테스트 예제

기본 L=pi에서 다음을 확인합니다.

| 입력 | 기대 결과 |
|---|---|
| `x` | a0≈0, a_n≈0, b_n≈2(-1)^(n+1)/n |
| `x**2` | a0≈pi²/3, b_n≈0 |
| `sin(x)` | b_1≈1, 나머지≈0 |
| `cos(x)` | a_1≈1, 나머지≈0 |
| `exp(x)` | 유한한 계수와 부분합, 주기 경계의 불연속 |
| `abs(x)` | 짝함수이므로 b_n≈0 |
| `sign(x)` | 홀수 b_n≈4/(pi n), 불연속점의 Gibbs 현상 |
| `3` | a0≈3, 나머지≈0, 상수 부분합 |

비교 N을 모두 해제하거나 현재 N=100, 비교 N=[1,3]으로 설정해도 동작합니다.
`sqrt(x)`, `log(x)`, 잘못된 수식은 오류 메시지로 처리합니다.
유한 표본으로 특이점 전체를 감지할 수는 없습니다. 표본 수를 늘려 결과의
안정성을 확인하세요. 불연속 함수는 표본 수에 따라 계수 오차가 달라집니다.
표시하는 MSE는 연속 적분 오차가 아니라 표본 오차 평균입니다.

## 구조와 향후 확장 계획

`app.py`는 입력/샘플링, 독립 수학 함수, Figure 생성, UI의 네 영역으로 구분됩니다.
계수 계산과 부분합 계산은 Streamlit 상태를 읽지 않는 독립 함수입니다.
필요할 때 이 함수들을 별도 모듈로 옮길 수 있습니다.

- v0.3: 기함수/우함수 판별, amplitude/phase spectrum, 표본 시각화,
  Gibbs 강조, 부분합 animation, predefined examples
- v0.4: DFT/FFT, sampling frequency, frequency bins, aliasing,
  Fourier Series와 DFT 비교

현재 spectrum은 개별 |a_n|, |b_n|이며 amplitude spectrum은 아직 구현하지 않았습니다.
DFT/FFT는 주기 경계 표본 중복 여부와 정규화를 별도 설계하여 추가할 예정입니다.
'''
(project_directory / 'README.md').write_text(readme_source, encoding='utf-8')


## 6. 설치 및 실행

터미널에서 위에 출력된 프로젝트 폴더로 이동한 뒤 아래 명령을 실행합니다.

```bash
pip install -r requirements.txt
streamlit run app.py
```

노트북 커널에도 패키지가 필요하면 아래 셀의 `%pip` 줄의 주석을 해제해서 실행하세요.
Streamlit 서버는 터미널에서 실행해야 노트북의 다음 셀이 막히지 않습니다.


In [ ]:
# 노트북 커널에 라이브러리가 없을 때만 실행합니다.
# %pip install streamlit "numpy>=2.0" "matplotlib>=3.9" "pandas>=2.2.2" "sympy>=1.13"


## 7. 수학 및 안정성 검증

모듈을 import하면 `main()`이 호출되지 않아 수학 함수만 시험할 수 있습니다.
`x`의 해석적 계수는 $b_n=2(-1)^{n+1}/n$입니다. 수치 적분이므로 허용 오차를 사용합니다.
`sign(x)`에서는 N=50일 때 양의 구간의 overshoot도 확인합니다.


In [ ]:
import importlib.util
import numpy as np

spec = importlib.util.spec_from_file_location("fourier_app", project_directory / "app.py")
app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app)

L = np.pi
x = np.linspace(-L, L, 20000)
a0, an, bn = app.calculate_fourier_coefficients(x, x, L, 100)
orders = np.arange(1, 101)
assert abs(a0) < 1e-12
assert np.max(np.abs(an)) < 1e-12
assert np.allclose(bn, 2 * (-1.0)**(orders + 1) / orders, atol=1e-5)
print("x: a0 =", a0, "max |a_n| =", np.max(np.abs(an)))

for text in ["x", "x**2", "sin(x)", "cos(x)", "exp(x)", "abs(x)", "sign(x)", "3"]:
    y = app.sample_function(app.parse_expression(text), x)
    assert y.shape == x.shape and np.all(np.isfinite(y))
    coefficients = app.calculate_fourier_coefficients(x, y, L, 100)
    result = app.calculate_fourier_sum(x, L, *coefficients, 100)
    assert np.all(np.isfinite(result))
    if text == "3":
        assert np.allclose(result, 3, atol=1e-12)
    if text == "sin(x)":
        assert np.isclose(coefficients[2][0], 1, atol=1e-12)
    if text == "cos(x)":
        assert np.isclose(coefficients[1][0], 1, atol=1e-12)
    print(text, "통과")

for comparison_N in [[], [1, 3], [1, 3, 5, 10, 20, 50]]:
    max_N = max([100, *comparison_N])
    coefficients = app.calculate_fourier_coefficients(x, x, L, max_N)
    assert len(coefficients[1]) >= 100
    app.calculate_fourier_sum(x, L, *coefficients, 100)

for invalid in ["sin(", "unknown(x)", "sqrt(x)", "log(x)", "1/0"]:
    try:
        app.sample_function(app.parse_expression(invalid), x)
    except Exception:
        pass
    else:
        raise AssertionError("잘못된 입력을 검출하지 못함: " + invalid)

y = app.sample_function(app.parse_expression("sign(x)"), x)
coefficients = app.calculate_fourier_coefficients(x, y, L, 50)
approximation = app.calculate_fourier_sum(x, L, *coefficients, 50)
peak = np.max(approximation[(x > 0) & (x < 0.3)])
assert 1.1 < peak < 1.25
print("Gibbs peak:", peak)
print("모든 수학 및 입력 안정성 검증 통과")


## 8. 그래프와 학습 실험

아래 그래프에서 불연속점 주변의 overshoot를 확인합니다.
앱에서 N을 10, 20, 50으로 늘리면 진동 영역이 좁아집니다.
오차 최대값은 불연속점과 주기 경계의 영향을 받으므로 항상 감소하지는 않습니다.


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

figure = app.make_comparison_figure(x, y, {50: approximation}, "Gibbs phenomenon: sign(x)")
display(figure)
plt.close(figure)


## 9. 직접 확인할 화면 조건과 확장 방향

1. 기본 `x`, `L=pi`에서 a0와 a_n이 거의 0인지 확인합니다.
2. 비교 N을 모두 지워도 왼쪽 그래프와 빈 수렴 표가 정상 표시되어야 합니다.
3. 현재 N=100, 비교 N=[1,3]에서 배열 부족 오류가 없어야 합니다.
4. `3` 입력 시 수평선, `sqrt(x)` 입력 시 빨간 오류 메시지가 표시되어야 합니다.
5. N과 표본 수를 바꾼 뒤 네 그래프, 계수 표, 오차 지표가 함께 갱신됩니다.

v0.3에는 대칭성 판별, amplitude/phase spectrum, 표본 표시, Gibbs 강조,
애니메이션, 예제 선택을 추가할 수 있습니다. v0.4에는 DFT/FFT 계산 함수를
별도로 만들고 주파수 축, sampling frequency, aliasing을 연결할 수 있습니다.
이번 버전에는 해당 기능들을 미리 구현하지 않았습니다.


## 10. 생성 후 검증 결과

NumPy 2.5.3 및 Streamlit 1.64.0 환경에서 검증했습니다.

- app.py와 모든 Python 셀의 문법 검사 통과
- 필수 7개 함수와 상수 함수 검증 통과
- `x`, `L=pi`: a0=0, 최대 |a_n| ≈ 1.89×10⁻¹⁵, 해석적 b_n 비교 통과
- `sign(x)`, N=50: 불연속점 오른쪽 peak ≈ 1.1791로 Gibbs overshoot 확인
- Streamlit AppTest에서 그래프 이미지 4개, 지표 3개, 표 2개 생성 확인
- 빈 비교 목록, 현재 N=100 > 비교 최대 N, 잘못된 수식 및 NaN/inf 처리 통과
- 입력 오류 후 정상 입력 복구 및 Figure 닫기 확인

화면 검증은 Streamlit AppTest 기반이며 실제 브라우저에서의 육안 검사는 수행하지 않았습니다.
